In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().parent.resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from TCT import (
    TCT,
    TCT_neighborhood_finder,
    Query_ARS,
    name_resolver,
    node_normalizer,
    translator_kpinfo,
    translator_metakg,
    translator_query,
)

In [2]:
from TCT.Query_ARS import ARS_neighborhood_finder, Query_ARS,check_ars_results


In [3]:
ARS_results = ARS_neighborhood_finder(
    node=['NCBIGene:2322'] ,  # replace with actual NodeInput instances
    neighbor_categories=['biolink:Drug'], 
    predicates=['biolink:related_to'] # optional edge predicates
)

In [4]:
query = {
  "message": {
    "query_graph": {
      "edges": {
        "e00": {
          "subject": "n00",
          "object": "n01",
          "predicates": ["biolink:related_to"]
        }
      },
      "nodes": {
        "n00": {
          "ids": [
            "MONDO:0005148"
          ]
        },
        "n01": {
          "categories": [
            "biolink:Drug"
          ]
        }
      }
    }
  },
  "submitter": "TCT"
}

In [5]:
import json
import requests
TEST_URL = "https://ars.ci.transltr.io/ars/api"
pk = Query_ARS(url=TEST_URL, json_file=query)

In [6]:
BASE_URL="https://ars.ci.transltr.io/ars/api"
merged_version_pk=check_ars_results(pk)


In [7]:
BASE_URL="https://ars.ci.transltr.io/ars/api"

def fetch_ars_results(merged_pk: str):
    url = BASE_URL+"/messages/"+merged_pk
    results = requests.get(url).json()
    return results

results = fetch_ars_results(merged_version_pk)

In [8]:
from TCT import ARS_pathfinder

# One-hop pathfinder over the ARS: every ARA is asked for paths between the
# two concepts, then the ARS merges and ranks their answers. Names resolve to
# CURIEs; at most one intermediate category is applied.
paths_result = ARS_pathfinder(
    "asthma",
    "albuterol",
    intermediate_categories=["Gene"],
)
len(paths_result.results), paths_result.merged_pk

(1, '36c801ac-81a3-4cc6-9994-c5c0c30984d2')

In [9]:
paths_result.summarize(5)

[{'rank': 1,
  'score': 100.0,
  'essence': 'result',
  'essence_category': None,
  'nodes': {'on': [{'id': 'CHEBI:2549',
     'name': 'Albuterol',
     'categories': ['biolink:Drug']}],
   'sn': [{'id': 'MONDO:0004979',
     'name': 'asthma',
     'categories': ['biolink:Disease']}]},
  'predicates': [],
  'primary_sources': [],
  'aras': ['infores:arax', 'infores:shepherd-aragorn'],
  'edge_count': 0}]

In [10]:
# The finders return an ARSResult, which extends FinderResult: the merged
# TRAPI message is available as .raw for the usual downstream consumers.
sorted(paths_result.knowledge_graph["nodes"])[:5]

['CHEBI:132742',
 'CHEBI:134556',
 'CHEBI:134557',
 'CHEBI:134558',
 'CHEBI:134559']